# 02 DOG2 clinical metadata and sample matching

Purpose: inspect the DOG2 clinical Excel file, identify endpoint columns, and match clinical samples to GSE238110 expression columns.

In [1]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

print("Project root:", PROJECT_ROOT)
print("Raw data dir:", RAW_DIR)
print("Processed data dir:", PROCESSED_DIR)

Project root: C:\Users\olegk\Desktop\paper4_sarcoma_dog
Raw data dir: C:\Users\olegk\Desktop\paper4_sarcoma_dog\data\raw
Processed data dir: C:\Users\olegk\Desktop\paper4_sarcoma_dog\data\processed


In [2]:
def find_file(folder, patterns):
    folder = Path(folder)
    hits = []
    for pattern in patterns:
        hits.extend(folder.glob(pattern))
    hits = sorted(set(hits))
    if not hits:
        raise FileNotFoundError(f"No matching files in {folder}. Patterns: {patterns}")
    if len(hits) > 1:
        print("Multiple candidates found:")
        for p in hits:
            print(" ", p.name)
        print("Using:", hits[0].name)
    return hits[0]

clinical_path = find_file(
    RAW_DIR / "canine_clinical_DOG2",
    ["*supplementary*table*s10*.xlsx", "*suppts10*.xlsx", "*.xlsx"]
)

print("Clinical file:", clinical_path)
xls = pd.ExcelFile(clinical_path)
print("Sheets:", xls.sheet_names)

Clinical file: C:\Users\olegk\Desktop\paper4_sarcoma_dog\data\raw\canine_clinical_DOG2\ccr-24-1854_supplementary_table_s10_suppts10.xlsx
Sheets: ['TME_subtype_SupplTable10']


In [3]:
sheets = {}
for sheet in xls.sheet_names:
    df = pd.read_excel(clinical_path, sheet_name=sheet)
    sheets[sheet] = df
    print(sheet, df.shape)
    display(df.head())

clinical_sheet = max(sheets, key=lambda s: sheets[s].shape[0] * sheets[s].shape[1])
clinical_raw = sheets[clinical_sheet].copy()

print("Selected sheet:", clinical_sheet)
print("Shape:", clinical_raw.shape)
display(pd.DataFrame({"column": clinical_raw.columns}))

TME_subtype_SupplTable10 (186, 16)


,Patient ID,Tumor Location,Site,age,weight,breed,gender,PH,ALP,Group,DFS_time,DFS_status,OS_time,OS_status,treatment,primary_immune_subtype
0,514,Left distal radius,UW,5.5,42.7,Mixed Breed,Spayed Female,0,Normal,NPHNALP,33,0,117,1,SOC,ID
1,518,Left proximal humerus,UW,8.3,44.3,Rottweiler,Spayed Female,1,Elevated,PHEALP,12,0,248,0,SOC,IE-ECM
2,619,Left distal tibia,OSU,7.5,30.8,Mixed Breed,Spayed Female,0,Normal,NPHNALP,63,1,622,0,SOC,IE-ECM
3,639,Left distal femur,OSU,6.9,36.2,Greyhound,Castrated Male,0,Normal,NPHNALP,234,1,283,1,SOC,ID
4,702,Right distal ulna,UIL,9.5,33.2,Labrador Retriever,Castrated Male,0,Normal,NPHNALP,153,1,171,0,SOC,IE-ECM


Selected sheet: TME_subtype_SupplTable10
Shape: (186, 16)


,column
0,Patient ID
1,Tumor Location
2,Site
3,age
4,weight
5,breed
6,gender
7,PH
8,ALP
9,Group


In [4]:
def normalize_id(x):
    x = "" if pd.isna(x) else str(x)
    x = x.strip()
    x = re.sub(r"\s+", "", x)
    x = x.replace("[", "").replace("]", "")
    return x.upper()

candidate_terms = ["sample", "dog", "patient", "case", "cotc", "id", "specimen", "rna", "seq", "tumor"]
candidate_id_cols = [
    c for c in clinical_raw.columns
    if any(term in str(c).lower() for term in candidate_terms)
]

print("Candidate ID columns:")
display(pd.DataFrame({"candidate_id_column": candidate_id_cols}))

for c in candidate_id_cols:
    vals = clinical_raw[c].dropna().astype(str).head(10).tolist()
    print("\n", c)
    print(vals)

Candidate ID columns:


,candidate_id_column
0,Patient ID
1,Tumor Location



 Patient ID
['514', '518', '619', '639', '702', '710', '716', '719', '720', '721']

 Tumor Location
['Left distal radius', 'Left proximal humerus', 'Left distal tibia', 'Left distal femur', 'Right distal ulna', 'Left distal femur', 'Left distal radius', 'Right distal radius', 'Right distal femur', 'Right proximal tibia']


In [5]:
sample_map_path = PROCESSED_DIR / "GSE238110_sample_id_map.csv"
sample_map = pd.read_csv(sample_map_path)
sample_map["norm_bracket_id"] = sample_map["sample_id_bracket"].map(normalize_id)
sample_map["norm_original_column"] = sample_map["original_sample_column"].map(normalize_id)

overlap_results = []

for c in candidate_id_cols:
    values = clinical_raw[c].map(normalize_id)
    overlap_bracket = values.isin(set(sample_map["norm_bracket_id"])).sum()
    overlap_original = values.isin(set(sample_map["norm_original_column"])).sum()
    overlap_results.append({
        "clinical_column": c,
        "overlap_with_bracket_ids": int(overlap_bracket),
        "overlap_with_original_columns": int(overlap_original),
        "non_missing": int(clinical_raw[c].notna().sum())
    })

overlap_df = pd.DataFrame(overlap_results).sort_values(
    ["overlap_with_bracket_ids", "overlap_with_original_columns", "non_missing"],
    ascending=False
)

display(overlap_df)
overlap_df.to_csv(PROCESSED_DIR / "DOG2_clinical_id_overlap_candidates.csv", index=False)

,clinical_column,overlap_with_bracket_ids,overlap_with_original_columns,non_missing
0,Patient ID,0,0,186
1,Tumor Location,0,0,186


In [6]:
# Edit this value if the automatic choice is wrong.
ID_COL = overlap_df.iloc[0]["clinical_column"]

clinical = clinical_raw.copy()
clinical["sample_id_norm"] = clinical[ID_COL].map(normalize_id)

sample_lookup = sample_map.copy()
sample_lookup["sample_id_norm"] = sample_lookup["norm_bracket_id"]

matched_clinical = clinical.merge(
    sample_lookup[["original_sample_column", "sample_id_bracket", "sample_id_norm"]],
    on="sample_id_norm",
    how="inner"
)

print("Using ID column:", ID_COL)
print("Matched samples:", matched_clinical.shape[0])
display(matched_clinical.head())

Using ID column: Patient ID
Matched samples: 0


,Patient ID,Tumor Location,Site,age,weight,breed,gender,PH,ALP,Group,DFS_time,DFS_status,OS_time,OS_status,treatment,primary_immune_subtype,sample_id_norm,original_sample_column,sample_id_bracket


In [7]:
endpoint_terms = [
    "survival", "dead", "death", "alive", "os", "overall", "dfi",
    "disease", "progress", "metast", "relapse", "event", "status",
    "time", "days", "months", "censor"
]

endpoint_cols = [
    c for c in matched_clinical.columns
    if any(term in str(c).lower() for term in endpoint_terms)
]

display(pd.DataFrame({"possible_endpoint_column": endpoint_cols}))

for c in endpoint_cols:
    print("\n", c)
    display(matched_clinical[c].value_counts(dropna=False).head(20))

,possible_endpoint_column
0,DFS_time
1,DFS_status
2,OS_time
3,OS_status



 DFS_time


Series([], Name: count, dtype: int64)


 DFS_status


Series([], Name: count, dtype: int64)


 OS_time


Series([], Name: count, dtype: int64)


 OS_status


Series([], Name: count, dtype: int64)

In [8]:
# Edit these after reviewing endpoint columns.
OS_TIME_COL = None
OS_EVENT_COL = None
DFI_TIME_COL = None
DFI_EVENT_COL = None
METASTASIS_COL = None

print("Set endpoint column variables after reviewing the columns above.")

Set endpoint column variables after reviewing the columns above.


In [9]:
clinical_export = matched_clinical.copy()

if OS_TIME_COL is not None:
    clinical_export["os_time"] = pd.to_numeric(clinical_export[OS_TIME_COL], errors="coerce")
if OS_EVENT_COL is not None:
    s = clinical_export[OS_EVENT_COL].astype(str).str.lower()
    clinical_export["os_event"] = np.where(s.str.contains("dead|deceased|event|yes|1|true"), 1,
                                  np.where(s.str.contains("alive|censor|no|0|false"), 0, np.nan))

if DFI_TIME_COL is not None:
    clinical_export["dfi_time"] = pd.to_numeric(clinical_export[DFI_TIME_COL], errors="coerce")
if DFI_EVENT_COL is not None:
    s = clinical_export[DFI_EVENT_COL].astype(str).str.lower()
    clinical_export["dfi_event"] = np.where(s.str.contains("event|progress|relapse|metast|yes|1|true"), 1,
                                   np.where(s.str.contains("censor|no|0|false"), 0, np.nan))

if METASTASIS_COL is not None:
    s = clinical_export[METASTASIS_COL].astype(str).str.lower()
    clinical_export["metastasis_event"] = np.where(s.str.contains("yes|metast|event|1|true"), 1,
                                          np.where(s.str.contains("no|0|false"), 0, np.nan))

clinical_export.to_csv(PROCESSED_DIR / "DOG2_clinical_matched_to_GSE238110.csv", index=False)

print("Saved matched clinical file.")
print(PROCESSED_DIR / "DOG2_clinical_matched_to_GSE238110.csv")

Saved matched clinical file.
C:\Users\olegk\Desktop\paper4_sarcoma_dog\data\processed\DOG2_clinical_matched_to_GSE238110.csv
